# Previsão de Resultados na Premier League: Superando a Média Estática com EMA e Poisson

Neste projeto, eu abordo um problema clássico na modelagem esportiva: **a dependência de médias estáticas**. 
Muitos apostadores e modelos básicos calculam a força de uma equipe usando a média de gols de toda a temporada. O problema? Uma equipe pode ter uma média alta, mas estar passando por uma crise recente de lesões ou de má forma.

**A Solução:** Implementar uma **Média Móvel Exponencial (EMA)** para dar um peso matemático maior (aprox. 86%) aos últimos 5 jogos da equipe. Em seguida, cruzar essa forma recente com a **Distribuição de Poisson** para prever as probabilidades exatas do próximo jogo e calcular as *Odds Justas* (Fair Lines).

---
### Passo 1: Extração e Preparação dos Dados
Vamos utilizar a base de dados pública do *Football-Data* para a Premier League (Temporada 23/24).

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import poisson

# 1. Carregar os dados reais da Premier League 
url = "https://www.football-data.co.uk/mmz4281/2324/E0.csv"
df = pd.read_csv(url)

# 2. Definir o confronto para o nosso Backtest (Manchester City vs Arsenal)
equipe_casa = 'Man City'
equipe_fora = 'Arsenal'

print(f"Base de dados carregada! Total de jogos analisados: {len(df)}")

Base de dados carregada! Total de jogos analisados: 380


### Passo 2: O Cálculo da Força Dinâmica (A Média Móvel Exponencial - EMA)
Em vez de usar `.mean()` para toda a base de dados, aplicamos a função `.ewm(span=5)` do Pandas. Isso atua como um fator de "esquecimento", garantindo que o algoritmo foque sua atenção na forma atual das equipes (os últimos confrontos).

In [24]:
# 1. Calcular as médias globais da Liga (a nossa régua de medição)
media_liga_casa = df['FTHG'].mean() # Gols marcados em casa na liga
media_liga_fora = df['FTAG'].mean() # Gols marcados fora na liga

# 2. Separar o histórico de cada equipe
df_casa = df[df['HomeTeam'] == equipe_casa].copy()
df_fora = df[df['AwayTeam'] == equipe_fora].copy()

# 3. Calcular a EMA (Força Atual) apenas com o valor do último jogo (.iloc[-1])
ema_ataque_casa = df_casa['FTHG'].ewm(span=5, adjust=False).mean().iloc[-1]
ema_defesa_casa = df_casa['FTAG'].ewm(span=5, adjust=False).mean().iloc[-1]

ema_ataque_fora = df_fora['FTAG'].ewm(span=5, adjust=False).mean().iloc[-1]
ema_defesa_fora = df_fora['FTHG'].ewm(span=5, adjust=False).mean().iloc[-1]

# 4. Ajustar as forças relativas (Equipe vs Liga)
forca_ataque_casa = ema_ataque_casa / media_liga_casa
forca_defesa_casa = ema_defesa_casa / media_liga_fora
forca_ataque_fora = ema_ataque_fora / media_liga_fora
forca_defesa_fora = ema_defesa_fora / media_liga_casa

# 5. Cálculo dos Gols Esperados (xG) para este jogo específico
xg_casa = forca_ataque_casa * forca_defesa_fora * media_liga_casa
xg_fora = forca_ataque_fora * forca_defesa_casa * media_liga_fora

print(f"xG Dinâmico (Expected Goals):")
print(f"{equipe_casa}: {xg_casa:.4f} gols")
print(f"{equipe_fora}: {xg_fora:.4f} gols")

xG Dinâmico (Expected Goals):
Man City: 0.9676 gols
Arsenal: 1.3154 gols


### Passo 3: O Motor de Probabilidades (Distribuição de Poisson)
Com os Gols Esperados (xG) em mãos, cruzamos os valores em uma Matriz Bivariada de Poisson. Simulamos todos os resultados possíveis (de 0x0 até 5x5) para extrair as probabilidades reais de Vitória, Empate e Derrota, gerando a *Odd Justa*.

In [26]:
# Definir limites da matriz (até 5 gols cobre a vasta maioria dos cenários)
max_gols = 6
prob_vitoria_casa = 0
prob_empate = 0
prob_vitoria_fora = 0

# Varrer a matriz de resultados possíveis
for i in range(max_gols):
    for j in range(max_gols):
        # Probabilidade independente de cada equipe marcar i ou j gols
        prob_i = poisson.pmf(i, xg_casa)
        prob_j = poisson.pmf(j, xg_fora)
        prob_placar = prob_i * prob_j
        
        # Agrupar as probabilidades nos respectivos "baldes"
        if i > j:
            prob_vitoria_casa += prob_placar
        elif i == j:
            prob_empate += prob_placar
        else:
            prob_vitoria_fora += prob_placar

print("PROBABILIDADES DO MODELO EMA:")
print(f"Vitória {equipe_casa}: {prob_vitoria_casa*100:.2f}%")
print(f"Empate:             {prob_empate*100:.2f}%")
print(f"Vitória {equipe_fora}:    {prob_vitoria_fora*100:.2f}%\n")

print(f"ODDS JUSTAS (Fair Lines):")
print(f"{equipe_casa}: {100/(prob_vitoria_casa*100):.2f}")
print(f"Empate:   {100/(prob_empate*100):.2f}")
print(f"{equipe_fora}:   {100/(prob_vitoria_fora*100):.2f}")

PROBABILIDADES DO MODELO EMA:
Vitória Man City: 27.41%
Empate:             27.94%
Vitória Arsenal:    44.37%

ODDS JUSTAS (Fair Lines):
Man City: 3.65
Empate:   3.58
Arsenal:   2.25
